# Validating single-cell embeddings with scIB

This notebook shows how to **query the embeddings of single-cell foundation models** with embpy and then **compare and validate** them with [scIB](https://scib.readthedocs.io/) metrics.

The workflow is:

1. Embed the same `AnnData` with several single-cell models via `BioEmbedder.embed(..., entity_type="cell")`. Each model lands in its own `.obsm` slot.
2. Score every embedding against a known cell-type label (and, optionally, a batch covariate) with `embpy.tl.compute_scib_metrics`.
3. Read off the per-model scIB scores and pick the embedding that best preserves biology (and, if relevant, removes batch effects).

## Requirements

The single-cell backbones (scGPT, Geneformer, UCE, ...) ship through the optional `helical` environment, and the metrics need the optional `scib` extra:

```bash
pip install "embpy[scib]"          # scib + scanpy (the metrics)
pixi install -e helical-gpu        # the single-cell model backbones
```

`pca` is always available as a lightweight classical baseline to compare against.

In [ ]:
import scanpy as sc

from embpy import BioEmbedder, tl

# A small annotated reference with a `cell_type` label (and ideally a `batch`
# covariate). Any AnnData with raw/normalised counts works; here we use one of
# scanpy's bundled datasets as a stand-in.
adata = sc.datasets.pbmc3k_processed()
adata.obs["cell_type"] = adata.obs["louvain"].astype(str)
adata

## 1. Query the single-cell model embeddings

We embed the *same* cells with several models. `embpy` handles the model-aware preprocessing; each call writes one matrix into `.obsm` under the `key` we pass.

In [ ]:
embedder = BioEmbedder(device="auto")

# Use whatever single-cell models are available in your environment.
# `pca` is the classical baseline and always present.
#
# These are registry keys, not family names. There is no bare
# "geneformer" key -- Geneformer ships as nine checkpoints
# (geneformer_v1_6L ... geneformer_v2_20L), and asking for the family
# name makes the guard below report "not available in this
# environment", which is a different and misleading claim from "that
# key does not exist".
models = ["pca", "scgpt", "geneformer_v2_12L", "uce"]

embedding_keys = []
for model in models:
    if model not in embedder.list_available_models():
        print(f"skip {model!r}: not available in this environment")
        continue
    key = f"X_{model}"
    adata = embedder.embed(
        adata,
        entity_type="cell",
        model=model,
        preprocessing="auto",
        output="anndata",
        key=key,
    )
    embedding_keys.append(key)

embedding_keys, list(adata.obsm.keys())

## 2. Score and compare with scIB

`compute_scib_metrics` builds a kNN graph on each embedding, optimises Leiden clustering against the label (for NMI/ARI), and computes the scIB metric battery. It returns one row per embedding, sorted by the aggregate `total` score (higher is better for every column).

Pass `batch_key=` as well when you have a batch covariate and care about batch correction; the `batch_correction` and `total` columns are filled in using scIB's standard 0.6 (bio) / 0.4 (batch) weighting.

In [ ]:
report = tl.compute_scib_metrics(
    adata,
    embedding_keys=embedding_keys,
    label_key="cell_type",
    batch_key=None,  # e.g. "batch" if your data has multiple batches
)
report

## 3. Read the result

- **Bio-conservation** (`nmi`, `ari`, `asw_label`, `isolated_label_asw`, `clisi`) measures how well the embedding keeps biologically distinct cell types apart.
- **Batch-correction** (`asw_batch`, `graph_conn`, `ilisi`, `kbet`) measures how well batches are mixed (only populated when `batch_key` is given).
- **`total`** is the scIB-weighted overall score used to rank embeddings.

The best single-cell embedding for this dataset is the top row:

In [ ]:
best = report.index[0]
print(f"Best embedding: {best}  (total={report.loc[best, 'total']:.3f})")
report[["bio_conservation", "batch_correction", "total"]]